# Global Development Lakehouse — Silver Layer Transformation

Reads raw Bronze Delta tables and produces cleaned, unpivoted Silver Delta tables:
- Unpivots wide year-columns (1960...2025) into rows (`country, year, value`)
- Filters out non-country aggregate rows (regions, income groups, "World")
- Cleans nulls and standardizes data types

In [0]:
df_bronze_gdp = spark.table("workspace.global_development.bronze_gdp_growth")
df_bronze_population = spark.table("workspace.global_development.bronze_population")
df_bronze_education = spark.table("workspace.global_development.bronze_education")

df_bronze_gdp.printSchema()

root
 |-- Country_Name: string (nullable = true)
 |-- Country_Code: string (nullable = true)
 |-- Indicator_Name: string (nullable = true)
 |-- Indicator_Code: string (nullable = true)
 |-- 1960: string (nullable = true)
 |-- 1961: double (nullable = true)
 |-- 1962: double (nullable = true)
 |-- 1963: double (nullable = true)
 |-- 1964: double (nullable = true)
 |-- 1965: double (nullable = true)
 |-- 1966: double (nullable = true)
 |-- 1967: double (nullable = true)
 |-- 1968: double (nullable = true)
 |-- 1969: double (nullable = true)
 |-- 1970: double (nullable = true)
 |-- 1971: double (nullable = true)
 |-- 1972: double (nullable = true)
 |-- 1973: double (nullable = true)
 |-- 1974: double (nullable = true)
 |-- 1975: double (nullable = true)
 |-- 1976: double (nullable = true)
 |-- 1977: double (nullable = true)
 |-- 1978: double (nullable = true)
 |-- 1979: double (nullable = true)
 |-- 1980: double (nullable = true)
 |-- 1981: double (nullable = true)
 |-- 1982: double (null

Right now this column structure:
Country_Name | Country_Code | Indicator_Name | Indicator_Code | 1960 | 1961 | ... | 2025 | _c70
I want this column structure:
Country_Name | Country_Code | year | value

In [0]:
# Identify which columns are actual year columns (numeric column names)
year_columns = [c for c in df_bronze_gdp.columns if c.isdigit()]

print(f"Found {len(year_columns)} year columns")
print(year_columns[:5], "...", year_columns[-5:])

Found 66 year columns
['1960', '1961', '1962', '1963', '1964'] ... ['2021', '2022', '2023', '2024', '2025']


In [0]:
# Build the stack() expression dynamically
# stack(n, 'col1_name', col1, 'col2_name', col2, ...) 
stack_expr = ", ".join([f"'{year}', `{year}`" for year in year_columns])

unpivot_expr = f"stack({len(year_columns)}, {stack_expr}) as (year, value)"

print(unpivot_expr[:200])  # just peek at the start to sanity check

stack(66, '1960', `1960`, '1961', `1961`, '1962', `1962`, '1963', `1963`, '1964', `1964`, '1965', `1965`, '1966', `1966`, '1967', `1967`, '1968', `1968`, '1969', `1969`, '1970', `1970`, '1971', `1971`


In [0]:
from pyspark.sql.functions import col

def unpivot_worldbank_df(df):
    """
    Takes a wide-format World Bank Bronze DataFrame (years as columns)
    and returns a long-format DataFrame: Country_Name, Country_Code, 
    Indicator_Name, Indicator_Code, year, value
    """
    year_columns = [c for c in df.columns if c.isdigit()]
    
    # Cast all year columns to double for consistent stacking
    df_fixed = df
    for year in year_columns:
        df_fixed = df_fixed.withColumn(year, col(f"`{year}`").cast("double"))
    
    stack_expr = ", ".join([f"'{year}', `{year}`" for year in year_columns])
    unpivot_expr = f"stack({len(year_columns)}, {stack_expr}) as (year, value)"
    
    df_unpivoted = df_fixed.selectExpr(
        "Country_Name", "Country_Code", "Indicator_Name", "Indicator_Code",
        unpivot_expr
    )
    
    return df_unpivoted

In [0]:
df_gdp_unpivoted = unpivot_worldbank_df(df_bronze_gdp)
df_population_unpivoted = unpivot_worldbank_df(df_bronze_population)
df_education_unpivoted = unpivot_worldbank_df(df_bronze_education)

df_gdp_unpivoted.show(5)
df_population_unpivoted.show(5)
df_education_unpivoted.show(5)

+--------------------+------------+--------------------+-----------------+----+-----+
|        Country_Name|Country_Code|      Indicator_Name|   Indicator_Code|year|value|
+--------------------+------------+--------------------+-----------------+----+-----+
|               Aruba|         ABW|GDP growth (annua...|NY.GDP.MKTP.KD.ZG|1960| NULL|
|Africa Eastern an...|         AFE|GDP growth (annua...|NY.GDP.MKTP.KD.ZG|1960| NULL|
|         Afghanistan|         AFG|GDP growth (annua...|NY.GDP.MKTP.KD.ZG|1960| NULL|
|Africa Western an...|         AFW|GDP growth (annua...|NY.GDP.MKTP.KD.ZG|1960| NULL|
|              Angola|         AGO|GDP growth (annua...|NY.GDP.MKTP.KD.ZG|1960| NULL|
+--------------------+------------+--------------------+-----------------+----+-----+
only showing top 5 rows
+--------------------+------------+-----------------+--------------+----+------------+
|        Country_Name|Country_Code|   Indicator_Name|Indicator_Code|year|       value|
+--------------------+------

In [0]:
#Load metadata + filter valid countries
from pyspark.sql.functions import col
df_country_metadata = spark.table("workspace.global_development.bronze_country_metadata")

df_valid_countries = df_country_metadata.filter(col("Region").isNotNull())

print("Valid country count:", df_valid_countries.count())

Valid country count: 230


In [0]:
def filter_valid_countries(df, valid_countries_df, dataset_name=""):
    """
    Inner-joins a dataset against the valid countries reference (from metadata)
    to keep only real countries, excluding aggregates like 'World', 'Arab World', etc.
    Prints before/after unique country counts for visibility.
    """
    before_count = df.select("Country_Code").distinct().count()
    
    df_filtered = df.join(
        valid_countries_df.select("Country_Code"),
        on="Country_Code",
        how="inner"
    )
    
    after_count = df_filtered.select("Country_Code").distinct().count()
    
    print(f"{dataset_name} - before: {before_count} | after: {after_count}")
    
    return df_filtered

In [0]:
df_gdp_countries_only = filter_valid_countries(df_gdp_unpivoted, df_valid_countries, "GDP")
df_population_countries_only = filter_valid_countries(df_population_unpivoted, df_valid_countries, "Population")
df_education_countries_only = filter_valid_countries(df_education_unpivoted, df_valid_countries, "Education")

GDP - before: 265 | after: 217
Population - before: 265 | after: 217
Education - before: 265 | after: 217


In [0]:
def cast_year_only(df, dataset_name=""):
    """
    Casts year to integer. Keeps NULL values as-is (no drop, no fill) —
    preserves full data completeness for coverage/reporting-gap analysis.
    """
    df_typed = df.withColumn("year", col("year").cast("int"))
    
    null_count = df_typed.filter(col("value").isNull()).count()
    total_count = df_typed.count()
    print(f"{dataset_name} (with nulls) - total: {total_count}, missing: {null_count} ({round(null_count/total_count*100, 1)}%)")
    
    return df_typed


def drop_nulls_and_cast_year(df, dataset_name=""):
    """
    Casts year to integer AND drops rows with NULL values —
    analysis-ready version for trend/correlation calculations.
    """
    df_clean = (
        df
        .filter(col("value").isNotNull())
        .withColumn("year", col("year").cast("int"))
    )
    
    print(f"{dataset_name} (nulls dropped) - row count: {df_clean.count()}")
    
    return df_clean

In [0]:
# Version 1: WITH nulls (for coverage/completeness analysis)
df_gdp_with_nulls = cast_year_only(df_gdp_countries_only, "GDP")
df_population_with_nulls = cast_year_only(df_population_countries_only, "Population")
df_education_with_nulls = cast_year_only(df_education_countries_only, "Education")

# Version 2: NULLS dropped (for trend/correlation analysis) - what you already had
df_gdp_clean = drop_nulls_and_cast_year(df_gdp_countries_only, "GDP")
df_population_clean = drop_nulls_and_cast_year(df_population_countries_only, "Population")
df_education_clean = drop_nulls_and_cast_year(df_education_countries_only, "Education")

GDP (with nulls) - total: 14322, missing: 2914 (20.3%)
Population (with nulls) - total: 14322, missing: 30 (0.2%)
Education (with nulls) - total: 14322, missing: 9197 (64.2%)
GDP (nulls dropped) - row count: 11408
Population (nulls dropped) - row count: 14292
Education (nulls dropped) - row count: 5125


In [0]:
#Drop redundant columns + rename value column (for both versions)
def finalize_columns(df, value_col_name):
    """Drops redundant indicator columns and renames 'value' to a meaningful name."""
    return (
        df
        .drop("Indicator_Name", "Indicator_Code")
        .withColumnRenamed("value", value_col_name)
    )

# Clean (nulls dropped) - analysis-ready
df_gdp_final = finalize_columns(df_gdp_clean, "gdp_growth_pct")
df_population_final = finalize_columns(df_population_clean, "population_total")
df_education_final = finalize_columns(df_education_clean, "education_expenditure_pct_gdp")

# With nulls - completeness/coverage version
df_gdp_with_nulls_final = finalize_columns(df_gdp_with_nulls, "gdp_growth_pct")
df_population_with_nulls_final = finalize_columns(df_population_with_nulls, "population_total")
df_education_with_nulls_final = finalize_columns(df_education_with_nulls, "education_expenditure_pct_gdp")

In [0]:
#Duplicate check function
def check_duplicates(df, key_cols, dataset_name=""):
    """
    Checks for duplicate rows based on key columns (e.g., Country_Code + year).
    Prints count of duplicate combinations found. Does not modify the DataFrame -
    this is a validation/reporting step only.
    """
    dup_count = (
        df.groupBy(*key_cols)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    
    status = "PASS - no duplicates" if dup_count == 0 else f"FAIL - {dup_count} duplicate combinations found"
    print(f"{dataset_name} duplicate check ({'+'.join(key_cols)}): {status}")
    
    return dup_count

In [0]:
import pyspark.sql.functions as F

check_duplicates(df_gdp_final, ["Country_Code", "year"], "GDP")
check_duplicates(df_population_final, ["Country_Code", "year"], "Population")
check_duplicates(df_education_final, ["Country_Code", "year"], "Education")

GDP duplicate check (Country_Code+year): PASS - no duplicates
Population duplicate check (Country_Code+year): PASS - no duplicates
Education duplicate check (Country_Code+year): PASS - no duplicates


0

In [0]:
# Outlier flagging — as a new column, not a separate list
def flag_outliers(df, value_col, lower_bound, upper_bound, dataset_name=""):
    """
    Adds an 'is_outlier' boolean flag column for values outside a plausible range.
    Does NOT drop rows - outliers may be genuine extreme events, not bad data.
    """
    df_flagged = df.withColumn(
        "is_outlier",
        (F.col(value_col) < lower_bound) | (F.col(value_col) > upper_bound)
    )
    
    outlier_count = df_flagged.filter(F.col("is_outlier") == True).count()
    print(f"{dataset_name}: {outlier_count} rows flagged as outliers (outside [{lower_bound}, {upper_bound}])")
    
    return df_flagged

In [0]:
# GDP growth: realistic range is roughly -50% to +50% (extreme but real cases exist, e.g. war, oil shocks)
df_gdp_final = flag_outliers(df_gdp_final, "gdp_growth_pct", -50, 50, "GDP")

# Population: no upper bound realistically needed, but should never be negative or zero
df_population_final = flag_outliers(df_population_final, "population_total", 1, float('inf'), "Population")

# Education expenditure: World Bank realistic range is roughly 0% to 20% of GDP
df_education_final = flag_outliers(df_education_final, "education_expenditure_pct_gdp", 0, 20, "Education")

GDP: 22 rows flagged as outliers (outside [-50, 50])
Population: 0 rows flagged as outliers (outside [1, inf])
Education: 1 rows flagged as outliers (outside [0, 20])


In [0]:
# Data Quality Check
print("=== SILVER LAYER DATA QUALITY SUMMARY ===\n")

for name, df in [("GDP", df_gdp_final), ("Population", df_population_final), ("Education", df_education_final)]:
    row_count = df.count()
    country_count = df.select("Country_Code").distinct().count()
    year_range = df.agg(F.min("year"), F.max("year")).collect()[0]
    outlier_count = df.filter(F.col("is_outlier") == True).count()
    
    print(f"{name}:")
    print(f"  Rows: {row_count}")
    print(f"  Unique countries: {country_count}")
    print(f"  Year range: {year_range[0]} - {year_range[1]}")
    print(f"  Outliers flagged: {outlier_count}")
    print()

=== SILVER LAYER DATA QUALITY SUMMARY ===

GDP:
  Rows: 11408
  Unique countries: 214
  Year range: 1961 - 2025
  Outliers flagged: 22

Population:
  Rows: 14292
  Unique countries: 217
  Year range: 1960 - 2025
  Outliers flagged: 0

Education:
  Rows: 5125
  Unique countries: 203
  Year range: 1970 - 2025
  Outliers flagged: 1



In [0]:
# Analysis-ready (nulls dropped) - already written earlier, re-confirming here
df_gdp_final.write.format("delta").mode("overwrite").saveAsTable("workspace.global_development.silver_gdp_growth")
df_population_final.write.format("delta").mode("overwrite").saveAsTable("workspace.global_development.silver_population")
df_education_final.write.format("delta").mode("overwrite").saveAsTable("workspace.global_development.silver_education")

# Completeness version (nulls preserved)
df_gdp_with_nulls_final.write.format("delta").mode("overwrite").saveAsTable("workspace.global_development.silver_gdp_growth_with_nulls")
df_population_with_nulls_final.write.format("delta").mode("overwrite").saveAsTable("workspace.global_development.silver_population_with_nulls")
df_education_with_nulls_final.write.format("delta").mode("overwrite").saveAsTable("workspace.global_development.silver_education_with_nulls")

print("All Silver tables (clean + with_nulls) created successfully")

All Silver tables (clean + with_nulls) created successfully


In [0]:
display(spark.sql("SHOW TABLES IN workspace.global_development"))

database,tableName,isTemporary
global_development,bronze_country_metadata,false
global_development,bronze_education,false
global_development,bronze_gdp_growth,false
global_development,bronze_population,false
global_development,silver_education,false
global_development,silver_education_with_nulls,false
global_development,silver_gdp_growth,false
global_development,silver_gdp_growth_with_nulls,false
global_development,silver_population,false
global_development,silver_population_with_nulls,false


In [0]:
display(spark.sql("select * from workspace.global_development.silver_population limit 10"))

Country_Code,Country_Name,year,population_total,is_outlier
ABW,Aruba,1960,54922.0,false
AFG,Afghanistan,1960,9035043.0,false
AGO,Angola,1960,5231654.0,false
ALB,Albania,1960,1608800.0,false
AND,Andorra,1960,9510.0,false
ARE,United Arab Emirates,1960,131334.0,false
ARG,Argentina,1960,2.0386045E7,false
ARM,Armenia,1960,1863705.0,false
ASM,American Samoa,1960,20133.0,false
ATG,Antigua and Barbuda,1960,55603.0,false


### Silver Layer Output
6 tables created: 3 analysis-ready (nulls dropped, outliers flagged) 
and 3 completeness-preserving (nulls retained) for GDP growth, 
population, and education expenditure.

## Incremental Load Pattern (Demo) — MERGE INTO

The Silver writes above use full `overwrite` mode, appropriate for this 
project since World Bank CSVs are downloaded as complete file replacements, 
not incremental feeds. In a real-world scenario where updated/revised 
records arrive periodically without a full reprocessing, `MERGE INTO` 
allows upserting only the changed/new records. Example (not executed, 
since no new incoming data exists in this project):

```python
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "workspace.global_development.silver_gdp_growth")

# df_new_gdp_data would represent incoming updated/new records
delta_table.alias("target").merge(
    df_new_gdp_data.alias("source"),
    "target.Country_Code = source.Country_Code AND target.year = source.year"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()
```